# T8.4 Bengali Indic Review & Sentiment Analyzer (Simple)

This notebook does 3 things:
1. Loads Bengali sentiment data from AI4Bharat IndicSentiment.
2. Fine-tunes `ai4bharat/indic-bert` for sentiment classification.
3. Creates a simple Streamlit app (`app.py`) for:
   - single review prediction (label + confidence)
   - CSV upload prediction + pie chart + top themes


In [1]:
# Run this once in a fresh environment
%pip install -q -U datasets==2.19.2 transformers==4.41.2 tokenizers==0.19.1 huggingface-hub==0.23.4 evaluate==0.4.2 accelerate==0.30.1 streamlit scikit-learn pandas matplotlib

# Login once in terminal if needed:
# huggingface-cli login

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
 )
import evaluate
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
MODEL_NAME = "ai4bharat/indic-bert"
LANG_CONFIG = "bengali"
OUT_DIR = "./indic_bert_bengali_sentiment"
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0505 20:54:19.393000 75577 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# Simple data loading
from datasets import Dataset, DatasetDict

# Train on test split, test on validation split
train_url = "https://huggingface.co/datasets/ai4bharat/IndicSentiment/resolve/main/data/test/bn.json"
test_url = "https://huggingface.co/datasets/ai4bharat/IndicSentiment/resolve/main/data/validation/bn.json"

train_df = pd.read_json(train_url, lines=True)
test_df = pd.read_json(test_url, lines=True)

# Keep only rows with text and label
train_df = train_df.dropna(subset=["INDIC REVIEW", "LABEL"]).copy()
test_df = test_df.dropna(subset=["INDIC REVIEW", "LABEL"]).copy()

# Build datasets
raw_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

text_col = "INDIC REVIEW"
label_col = "LABEL"

train_ds = raw_ds["train"]
test_ds = raw_ds["test"]

# Automatically includes Negative/Neutral/Positive if present
label_names = sorted(list(set(train_ds[label_col])))
num_labels = len(label_names)

print("labels:", label_names)
print("train rows:", len(train_ds), "test rows:", len(test_ds))


labels: ['Negative', 'Positive']
train rows: 998 test rows: 156


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

id2label = {i: str(lbl) for i, lbl in enumerate(label_names)}
label2id = {str(lbl): i for i, lbl in id2label.items()}


def preprocess(batch):
    tokens = tokenizer(batch[text_col], truncation=True, max_length=128)
    tokens["labels"] = [label2id[str(x)] for x in batch[label_col]]
    return tokens

train_tok = train_ds.map(preprocess, batched=True)
test_tok = test_ds.map(preprocess, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision": metric_precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": metric_recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
 )

args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=SEED,
 )

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

trainer.train()
metrics = trainer.evaluate()
print("Test metrics:", metrics)

pred_out = trainer.predict(test_tok)
preds = np.argmax(pred_out.predictions, axis=-1)
cm = confusion_matrix(pred_out.label_ids, preds)
print("Confusion matrix:\n", cm)
print(
    "Classification report:\n",
    classification_report(
        pred_out.label_ids,
        preds,
        target_names=[id2label[i] for i in range(num_labels)],
    ),
)

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved model at:", OUT_DIR)

/opt/homebrew/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 156/156 [00:00<00:00, 27807.54 examples/s]
Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/homebrew/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/504 [00:00<?, ?it/s]/opt/homebrew/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: Us

{'eval_loss': 0.6204957962036133, 'eval_accuracy': 0.6923076923076923, 'eval_f1': 0.6716967243283032, 'eval_precision': 0.7396449704142012, 'eval_recall': 0.6923076923076923, 'eval_runtime': 1.6865, 'eval_samples_per_second': 92.498, 'eval_steps_per_second': 5.929, 'epoch': 1.0}


/opt/homebrew/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                 
 25%|██▌       | 126/504 [01:03<02:39,  2.37it/s]

{'eval_loss': 0.5010839104652405, 'eval_accuracy': 0.7692307692307693, 'eval_f1': 0.7665312628547921, 'eval_precision': 0.7766642330921882, 'eval_recall': 0.7692307692307693, 'eval_runtime': 1.792, 'eval_samples_per_second': 87.054, 'eval_steps_per_second': 5.58, 'epoch': 2.0}


/opt/homebrew/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                 
 38%|███▊      | 189/504 [01:54<04:17,  1.22it/s]

...

{'train_runtime': 457.3945, 'train_samples_per_second': 17.455, 'train_steps_per_second': 1.102, 'train_loss': 0.3453787389019179, 'epoch': 8.0}


100%|██████████| 10/10 [00:03<00:00,  3.01it/s]
/opt/homebrew/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test metrics: {'eval_loss': 0.6047645807266235, 'eval_accuracy': 0.8269230769230769, 'eval_f1': 0.826873192686179, 'eval_precision': 0.8268913848182142, 'eval_recall': 0.8269230769230769, 'eval_runtime': 3.5678, 'eval_samples_per_second': 43.724, 'eval_steps_per_second': 2.803, 'epoch': 8.0}


100%|██████████| 10/10 [00:02<00:00,  3.51it/s]

Confusion matrix:
 [[68 13]
 [14 61]]
Classification report:
               precision    recall  f1-score   support

    Negative       0.83      0.84      0.83        81
    Positive       0.82      0.81      0.82        75

    accuracy                           0.83       156
   macro avg       0.83      0.83      0.83       156
weighted avg       0.83      0.83      0.83       156

Saved model at: ./indic_bert_bengali_sentiment


In [4]:
# Simple single review test
clf = pipeline("text-classification", model=OUT_DIR, tokenizer=OUT_DIR, top_k=None)

sample_text = "এই বইটা দারুণ, আমার খুব ভালো লেগেছে।"
all_scores = clf(sample_text)[0]
best = max(all_scores, key=lambda x: x["score"])

print("Review:", sample_text)
print("Predicted label:", best["label"])
print("Confidence:", round(float(best["score"]), 4))
print("All scores:", all_scores)


Review: এই বইটা দারুণ, আমার খুব ভালো লেগেছে।
Predicted label: Positive
Confidence: 0.9694
All scores: [{'label': 'Positive', 'score': 0.9693635106086731}, {'label': 'Negative', 'score': 0.030636508017778397}]


In [5]:
app_code = r'''
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline

MODEL_DIR = "./indic_bert_bengali_sentiment"
st.set_page_config(page_title="Bengali Sentiment Analyzer", layout="wide")
st.title("Bengali Review Sentiment Analyzer")

@st.cache_resource
def load_model():
    return pipeline("text-classification", model=MODEL_DIR, tokenizer=MODEL_DIR, top_k=None)

clf = load_model()

def predict_one(text):
    scores = clf(text)[0]
    best = max(scores, key=lambda x: x["score"])
    return best["label"], float(best["score"])

st.subheader("Single Review")
text_input = st.text_area("Paste one Bengali review", "")
if st.button("Predict"):
    if text_input.strip():
        lbl, conf = predict_one(text_input.strip())
        st.success(f"Label: {lbl} | Confidence: {conf:.2f}")
    else:
        st.warning("Please enter a review.")

st.divider()
st.subheader("Bulk CSV Analysis")
st.write("Upload a CSV with a text column (e.g., review).")
file = st.file_uploader("Upload CSV", type=["csv"])

if file is not None:
    df = pd.read_csv(file)
    st.write("Columns:", list(df.columns))

    text_col = st.selectbox("Select text column", df.columns)

    if st.button("Run Bulk Prediction"):
        texts = df[text_col].fillna("").astype(str).tolist()
        labels, confs = [], []
        for t in texts:
            lbl, c = predict_one(t)
            labels.append(lbl)
            confs.append(c)

        out = df.copy()
        out["pred_label"] = labels
        out["confidence"] = confs

        st.dataframe(out.head(20))

        st.subheader("Sentiment Distribution")
        counts = out["pred_label"].value_counts()
        fig, ax = plt.subplots()
        ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%")
        ax.set_title("Predicted Sentiment")
        st.pyplot(fig)

        st.subheader("Top Themes (Simple Keywords)")
        vec = CountVectorizer(max_features=20, ngram_range=(1, 2), stop_words=None)
        X = vec.fit_transform(out[text_col].fillna("").astype(str))
        vocab = vec.get_feature_names_out()
        freqs = X.sum(axis=0).A1
        themes = pd.DataFrame({"theme": vocab, "count": freqs}).sort_values("count", ascending=False)
        st.dataframe(themes.head(10))

        st.download_button(
            "Download Predictions CSV",
            out.to_csv(index=False).encode("utf-8"),
            file_name="bengali_sentiment_predictions.csv",
            mime="text/csv",
        )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created app.py")


Created app.py


## Run the Streamlit app

After running all cells above:

```bash
streamlit run app.py
```

Then open the local URL shown in terminal (usually `http://localhost:8501`).
